In [11]:
%pip install sentence_transformers
%pip install openai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached openai-2.21.0-py3-none-any.whl.metadata (29 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached jiter-0.13.0-cp312-cp312-win_amd64.whl.metadata (5.3 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
Using cached openai-2.21.0-py3-none-any.whl (1.1 MB)
Using cached distro-1.9.0-py3-none-any.whl (20 kB)
Using cached jiter-0.13.0-cp312-cp312-win_amd64.whl (205 kB)
Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
%pip install scikit-learn

# Step 1: Define Sample Documents
documents = [
    {"section": "Employee Info", "content": "John's pay is processed on the 1st of every month."},
    {"section": "Employee Info", "content": "Mark is on a leave of absence until next Monday."},
    {"section": "Employee Info", "content": "Julie is a software engineer."},
    {"section": "Employee Info", "content": "Julie's pay is processed on the 1st of every month."},
    {"section": "Employee Info", "content": "Mark is a product manager."},
    {"section": "Employee Info", "content": "John is an AI architect and has salary of 500K USD."},
]

# Step 2: Get Content Texts
content_corpus = [doc["content"] for doc in documents]
content_corpus

from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
doc_vectors = model.encode(content_corpus)

doc_vectors
print(doc_vectors.shape)



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


c:\Users\nirma\Training\GenAI_Projects\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 541.17it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


(6, 384)


In [12]:
import os
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv(override=True, dotenv_path="../.env.local")
my_api_key = os.getenv("OPENAI_API_KEY")
print(f"API_KEY: {my_api_key[:5]}")

my_client = OpenAI(api_key=my_api_key)
# my_client


# Define your target function that performs retrieval per-question
def ask_question_open_ai(prompt, context=''):
    """Call the LLM with the provided prompt and context.

    IMPORTANT: use the passed-in prompt (not a global variable) so each
    evaluation example can be answered correctly.
    """
    llm_response = my_client.chat.completions.create(
        model="gpt-5-nano",
        messages=[
            {"role": "system", "content": '''
             You are an assistant who answers only based on the given context.
             '''},
            {"role": "user", "content": f"Context: {context}\n\nUser Question: {prompt}"}
        ]

    )
    return llm_response.choices[0].message.content
    

API_KEY: sk-pr


In [13]:
# from langchain_openai import ChatOpenAI
from langsmith import traceable # Need to enable tracing on LangSmith

# Define your target function that performs retrieval per-question
@traceable 
def ask_question(inputs):
    question = inputs["question"]
    # compute embedding for the question
    query_vec = model.encode([question])[0]

    # compute cosine similarities between query and doc_vectors
    similarities = model.similarity(query_vec, doc_vectors)

    import numpy as np

    # # Ensure it's a 1D numpy array
    similarities = np.asarray(similarities).squeeze()

    # Now get top 3
    top_3_indices = np.argsort(similarities)[::-1][:3]
    top_scores = similarities[top_3_indices]
    top_scores

    top_docs = [documents[i]['content'] for i in top_3_indices]

    # # pick top-3 supporting docs and build context
    # top_3_indices = np.argsort(sims)[::-1][:3]
    top_docs = [content_corpus[i] for i in top_3_indices]
    context = "\n---\n".join(top_docs)

    # call LLM with question and its retrieved context
    answer = ask_question_open_ai(question, context)
    
    return {"answer": answer}
    

In [14]:
ask_question({"question": "When is John's pay processed?"})


{'answer': 'On the 1st of every month.'}

In [15]:
from langsmith import Client

client = Client()

dataset_name = "2025Dec-Employee-Info-QA-Dataset-5"
dataset = client.create_dataset(dataset_name=dataset_name)

examples = [
    {"input": "When is John's pay processed?", "output": "John's pay is processed on the 1st of every month."},
    {"input": "What is Julie's job title?", "output": "Julie is a software engineer."},
    {"input": "What is John's salary?", "output": "John has a salary of 500K USD."},
    {"input": "What is Mark's current work status?", "output": "Mark is on a leave of absence until next Monday."},
]

for ex in examples:
    client.create_example(inputs={"question": ex["input"]}, outputs={"answer": ex["output"]}, dataset_id=dataset.id)

print(f" Dataset '{dataset_name}' created with {len(examples)} examples.")


 Dataset '2025Dec-Employee-Info-QA-Dataset-5' created with 4 examples.


In [23]:

import os
import json
from openai import OpenAI
from langsmith.evaluation import RunEvaluator

class SimpleCorrectness(RunEvaluator):
    """LLM-as-a-judge correctness evaluator (version-safe)."""

    def evaluate_run(self, run, example, **kwargs):

        question = example.inputs.get("question", "")
        reference = example.outputs.get("answer", "")
        prediction = run.outputs.get("answer")
        
       
        client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

        response = client.chat.completions.create(
            model="gpt-5-nano",
            messages=[
                {
                    "role": "system",
                    "content": "You are a semantic correctness evaluator."
                },
                {
                    "role": "user",
                    "content": f"""
                        Question: {question}
                        Reference answer: {reference}
                        Model prediction: {prediction}

                        Return JSON only:
                        {{"score": <number between 0 and 1>, "reason": "<short explanation>"}}
                        """
                }
            ],
        )

        content = response.choices[0].message.content
        data = json.loads(content)

        score = float(data["score"])
        reason = data["reason"]

        # return max(0.0, min(1.0, score)), reason
        return {
            "key": "correctness",
            "score": max(0, min(1, score)),
            "comment": reason,
        }
        


In [24]:
from langsmith.evaluation import evaluate

simple_correctness = SimpleCorrectness()

results = evaluate(
    ask_question, # the function to evaluate
    data=dataset_name, # the dataset to use
    evaluators=[simple_correctness], # the evaluators to use
    experiment_prefix="langsmith_eval_test",
)


View the evaluation results for experiment: 'langsmith_eval_test-2c4cfea6' at:
https://smith.langchain.com/o/5bd1dd07-2539-435d-8756-3fd33e2ac54b/datasets/1456b8cb-e770-4805-8fe8-4ca638f614d5/compare?selectedSessions=eeaf3c1d-85f2-4a0b-b5cd-347bc0828630




4it [00:30,  7.52s/it]


In [18]:

for r in results:  
    print (r)


{'run': RunTree(id=019c7469-4549-74e2-b969-a890bdf0968a, name='ask_question', run_type='chain', dotted_order='20260219T053942025422Z019c7469-4549-74e2-b969-a890bdf0968a'), 'example': <class 'langsmith.schemas.Example'>(id=6a62e8b0-cb2b-4bb0-87d0-5d4616d4dfbc, dataset_id=1456b8cb-e770-4805-8fe8-4ca638f614d5, link='https://smith.langchain.com/o/5bd1dd07-2539-435d-8756-3fd33e2ac54b/datasets/1456b8cb-e770-4805-8fe8-4ca638f614d5/e/6a62e8b0-cb2b-4bb0-87d0-5d4616d4dfbc'), 'evaluation_results': {'results': []}}
{'run': RunTree(id=019c7469-6368-7ac1-98a9-c61cb9494ed8, name='ask_question', run_type='chain', dotted_order='20260219T053949736426Z019c7469-6368-7ac1-98a9-c61cb9494ed8'), 'example': <class 'langsmith.schemas.Example'>(id=3f78587f-7bfc-4204-b990-3dd23ccc9ee6, dataset_id=1456b8cb-e770-4805-8fe8-4ca638f614d5, link='https://smith.langchain.com/o/5bd1dd07-2539-435d-8756-3fd33e2ac54b/datasets/1456b8cb-e770-4805-8fe8-4ca638f614d5/e/3f78587f-7bfc-4204-b990-3dd23ccc9ee6'), 'evaluation_results':

In [19]:
for r in results:   # each r is a dict
    example = r["example"]
    eval_results = r["evaluation_results"]["results"]
    run = r["run"]

    print(f"Question: {example.inputs['question']}")
    print(f"Expected: {example.outputs['answer']}")

    # Extract model output
    
    if hasattr(run, "outputs") and "answer" in run.outputs:
        print(f"Predicted: {run.outputs['answer']}")
    else:
        print("Predicted: (no output found)")

    # Print evaluator results
    for e in eval_results:
        # print(e)
        print(f"Evaluator: {e.key}, Score: {e.score}, Explanation: {getattr(e, 'reason', None)}")

Question: What is Mark's current work status?
Expected: Mark is on a leave of absence until next Monday.
Predicted: He is currently on a leave of absence until next Monday.
Question: What is John's salary?
Expected: John has a salary of 500K USD.
Predicted: John's salary is 500K USD.
Question: What is Julie's job title?
Expected: Julie is a software engineer.
Predicted: Software engineer.
Question: When is John's pay processed?
Expected: John's pay is processed on the 1st of every month.
Predicted: John's pay is processed on the 1st of every month.
